# KO comparison: MaskGIT vs JEPA (IL1B @ 90m → 6h / 10h)

Numerical focus on the **left of the cytokine-signaling boxplot** lymphoid arm.

Dump labels (no `gamma-delta T cell` in `lps_90min_perturb` KO dumps):

| boxplot label | dump `cell_type_harmonized` | unique n @6h (approx.) |
|---|---|---|
| NKT cells | `NKT` | ~16 (thin — treat cautiously) |
| gamma-delta T cell | **absent** | — |
| NK cells | `NK` | ~187 |
| *(substitute 3rd)* | `CD8+ T cells` | ~185 |

JEPA dumps use `--ko-resync-quiz`. Registry: `configs/ko/comparison.yaml`. Figures → `ko_comparison_res_90m_jepa100_qresync`. Keep `RERUN_INFERENCE = False`.


## 0. What is comparable

Same tokenization, IL1B source edit at 90m, predict 6h/10h. Contrast = KO − control (not vs true).

- MaskGIT: token-sequence mask KO.
- JEPA: source pad/mask + frozen CountHead; qresync refreshes query `src_position`.
- `source_only` = honest path; `copy_target` = leaky quiz from target tokens.

Primary readouts below: **IL1B self Δ**, **marker Δ table**, **Spearman / sign agreement vs MaskGIT** — per focus cell type.


## 1. Load YAML registry


In [ ]:
from __future__ import annotations

from pathlib import Path
import pickle
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
from IPython.display import display
from scipy import stats

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_context("talk", font_scale=0.9)

REPO = Path("/home/stuke1/perturbgen/Perturbgen")
KO_CFG_DIR = REPO / "docs" / "examples" / "configs" / "ko"
CMP_YAML = KO_CFG_DIR / "comparison.yaml"

RERUN_INFERENCE = False
LOAD_H5AD = False
MIN_CELLS = 15  # NKT is ~16; below this skip the CT

with open(CMP_YAML) as f:
    cmp = yaml.safe_load(f)

def load_exp(name: str) -> dict:
    path = KO_CFG_DIR / cmp["experiments"][name]["yaml"]
    with open(path) as f:
        cfg = yaml.safe_load(f)
    cfg["_yaml_path"] = str(path)
    cfg["_name"] = name
    return cfg

EXPS = {k: load_exp(k) for k in cmp["experiments"]}
REF_NAME = cmp.get("reference") or next(iter(EXPS))
PAIRS = [tuple(p) for p in (cmp.get("pairs") or [])]
GENE = cmp["gene_ensembl"]
GENE_SYM = cmp["gene_symbol"]
FOCUS_CTS = list(cmp.get("focus_cell_types") or [cmp["focus_cell_type"]])
TIMES = list(cmp["times"])
MARKERS = [g.upper() for g in cmp["cytokine_markers"]]
OUT_DIR = Path(cmp["output_dir"])
OUT_DIR.mkdir(parents=True, exist_ok=True)

def exp_lab(name: str) -> str:
    return EXPS[name]["experiment"]["label"]

print("registry:", CMP_YAML)
print("focus CTs:", FOCUS_CTS)
print(cmp.get("focus_cell_types_note", ""))
for name, cfg in EXPS.items():
    exp = cfg["experiment"]
    print(
        f"  {name:22s}  {exp['label']:28s}  "
        f"mode={exp.get('perturbation_mode','?'):4s}  "
        f"resync={exp.get('ko_resync_quiz')}  "
        f"dump={cfg['trainer']['output_dir']}"
    )
print("figures ->", OUT_DIR)


## 2. File inventory


In [ ]:
def dump_dir(cfg: dict) -> Path:
    return Path(cfg["trainer"]["output_dir"])


def list_dumps(cfg: dict) -> list[Path]:
    d = dump_dir(cfg)
    g = cfg["experiment"].get("dump_glob", "*_minference_adata_gENSG00000125538_ssrc_*.h5ad")
    return sorted(d.glob(g))


def preferred_h5ad(cfg: dict) -> Path | None:
    d = dump_dir(cfg)
    pref = cfg["experiment"].get("preferred_dump")
    if pref and (d / pref).is_file():
        return d / pref
    hits = list_dumps(cfg)
    return hits[-1] if hits else None


rows = []
for name, cfg in EXPS.items():
    d = dump_dir(cfg)
    h5 = preferred_h5ad(cfg)
    exp = cfg["experiment"]
    rows.append({
        "name": name,
        "experiment": exp["label"],
        "ko_replace": exp.get("perturbation_mode"),
        "count_input": exp.get("count_input"),
        "resync": exp.get("ko_resync_quiz"),
        "h5ad": h5.name if h5 else None,
        "n_h5ad": len(list_dumps(cfg)),
    })
inv = pd.DataFrame(rows)
display(inv)
print("dumps ready" if inv["n_h5ad"].min() >= 1 else "missing h5ad")


## 3. Maps + effect tables (per focus cell type)

Builds / caches `il1b_ko_effect_{time}_{celltype}_dedup.csv` under each dump dir.


In [ ]:
with open(cmp["maps"]["token_id_to_ensembl"], "rb") as f:
    TOKEN_TO_ENS = {int(k): str(v) for k, v in pickle.load(f).items()}
with open(cmp["maps"]["symbol_to_ensembl"], "rb") as f:
    SYM_TO_ENS = pickle.load(f)
ENS_TO_SYM = {str(v): str(k).upper() for k, v in SYM_TO_ENS.items()}

print("token->ensembl", len(TOKEN_TO_ENS), "| IL1B", SYM_TO_ENS.get("IL1B"))


def remap_effect(df: pd.DataFrame, family: str) -> pd.DataFrame:
    out = df.copy()
    raw = out["ensembl_id"].astype(str)
    if raw.str.fullmatch(r"\d+").mean() > 0.8:
        out["ensembl_id"] = raw.map(lambda x: TOKEN_TO_ENS.get(int(x)) if str(x).isdigit() else x)
    out["gene_symbol"] = out["ensembl_id"].map(
        lambda e: ENS_TO_SYM.get(str(e), str(e) if e is not None else e)
    )
    return out[out["ensembl_id"].notna()].copy()


def bh_fdr(pvals: np.ndarray) -> np.ndarray:
    p = np.asarray(pvals, dtype=float)
    order = np.argsort(p)
    ranked = p[order]
    n = len(p)
    adjusted = ranked * n / np.arange(1, n + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    out = np.empty_like(adjusted)
    out[order] = np.clip(adjusted, 0, 1)
    return out


def effect_table_from_h5ad(cfg: dict, time_label: str, cell_type: str, eps: float = 1e-3) -> pd.DataFrame:
    import anndata as ad

    h5 = preferred_h5ad(cfg)
    if h5 is None:
        raise FileNotFoundError(cfg["experiment"]["label"])
    adata = ad.read_h5ad(h5)
    if "pert_counts" not in adata.layers:
        adata.layers["pert_counts"] = np.asarray(adata.X).copy()
    mask = adata.obs["time_after_LPS"].astype(str) == time_label
    mask &= adata.obs["cell_type_harmonized"].astype(str) == cell_type
    sub = adata[mask]
    if sub.n_obs == 0:
        raise ValueError(f"No cells for {cell_type} @ {time_label}")
    if "cell_pairing_index" in sub.obs.columns:
        keep_unique = ~sub.obs["cell_pairing_index"].astype(str).duplicated(keep="first").values
        if not keep_unique.all():
            print(f"  {cfg['experiment']['label']} {cell_type} {time_label}: {sub.n_obs} -> {int(keep_unique.sum())}")
            sub = sub[keep_unique]
    if sub.n_obs < MIN_CELLS:
        raise ValueError(f"only {sub.n_obs} unique cells for {cell_type} @ {time_label} (<{MIN_CELLS})")

    pert = np.asarray(sub.layers["pert_counts"], dtype=np.float64)
    ctrl = np.asarray(sub.layers["pred_counts"], dtype=np.float64)
    true = np.asarray(sub.layers["true_counts"], dtype=np.float64)
    mean_pert, mean_ctrl, mean_true = pert.mean(0), ctrl.mean(0), true.mean(0)
    delta = mean_pert - mean_ctrl
    log2fc = np.log2(mean_pert + eps) - np.log2(mean_ctrl + eps)
    _, pvals = stats.ttest_rel(pert, ctrl, axis=0, nan_policy="omit")
    pvals = np.nan_to_num(pvals, nan=1.0)
    gene_ids = sub.var_names.astype(str)
    return pd.DataFrame({
        "ensembl_id": gene_ids,
        "gene_symbol": [ENS_TO_SYM.get(g, g) for g in gene_ids],
        "mean_pert": mean_pert,
        "mean_ctrl": mean_ctrl,
        "mean_true": mean_true,
        "delta": delta,
        "log2fc": log2fc,
        "pvalue": pvals,
        "fdr": bh_fdr(pvals),
        "n_cells": sub.n_obs,
    }).sort_values("delta")


def ensure_effect_csv(cfg: dict, time_label: str, cell_type: str) -> Path:
    d = dump_dir(cfg)
    tag = cell_type.replace(" ", "_").replace("+", "plus")
    path = d / f"il1b_ko_effect_{time_label}_{tag}_dedup.csv"
    if path.is_file():
        return path
    df = effect_table_from_h5ad(cfg, time_label, cell_type)
    df.to_csv(path, index=False)
    print("wrote", path, "n_cells=", int(df["n_cells"].iloc[0]))
    return path


def load_effect(cfg: dict, time_label: str, cell_type: str) -> pd.DataFrame:
    path = ensure_effect_csv(cfg, time_label, cell_type)
    df = remap_effect(pd.read_csv(path), cfg["experiment"]["family"])
    df["time"] = time_label
    df["cell_type"] = cell_type
    df["experiment"] = cfg["experiment"]["label"]
    df["family"] = cfg["experiment"]["family"]
    df["name"] = cfg["_name"]
    return df


effects: dict[tuple[str, str, str], pd.DataFrame] = {}
skipped = []
for ct in FOCUS_CTS:
    for name, cfg in EXPS.items():
        for tp in TIMES:
            try:
                effects[(name, ct, tp)] = load_effect(cfg, tp, ct)
                df = effects[(name, ct, tp)]
                n = int(df["n_cells"].iloc[0]) if "n_cells" in df.columns else len(df)
                print(f"{name:22s} {ct:16s} {tp:8s}  genes={len(df)}  n_cells={n}  IL1B={GENE in set(df.ensembl_id)}")
            except Exception as e:
                skipped.append((name, ct, tp, str(e)))
                print(f"SKIP {name} {ct} {tp}: {e}")

if skipped:
    print("\nskipped", len(skipped), "combos")


## 4. IL1B self-effect (numbers)

`delta = mean(KO) − mean(control)` for IL1B only.


In [ ]:
il1b_rows = []
for (name, ct, tp), df in effects.items():
    hit = df.loc[df["ensembl_id"] == GENE]
    if hit.empty:
        hit = df.loc[df["gene_symbol"].astype(str).str.upper() == "IL1B"]
    if hit.empty:
        continue
    r = hit.iloc[0]
    il1b_rows.append({
        "experiment": exp_lab(name),
        "name": name,
        "cell_type": ct,
        "time": tp.replace("_LPS", ""),
        "n_cells": int(r["n_cells"]) if "n_cells" in r.index and pd.notna(r["n_cells"]) else np.nan,
        "mean_pert": r["mean_pert"],
        "mean_ctrl": r["mean_ctrl"],
        "mean_true": r["mean_true"],
        "delta": r["delta"],
        "log2fc": r["log2fc"],
        "fdr": r["fdr"],
    })
il1b_tab = pd.DataFrame(il1b_rows)
display(il1b_tab.round(4))
il1b_tab.to_csv(OUT_DIR / "il1b_self_effect_by_celltype.csv", index=False)

# compact pivot: delta only @ 6h
sub6 = il1b_tab[il1b_tab["time"] == "6h"]
if len(sub6):
    piv = sub6.pivot_table(index="cell_type", columns="experiment", values="delta")
    display(piv.round(4))
    piv.to_csv(OUT_DIR / "il1b_delta_6h_by_celltype.csv")


## 5. Marker panel (numbers)

Same cytokine markers as the registry, Δ per experiment × cell type × time.


In [ ]:
want_ens = {SYM_TO_ENS.get(s) for s in MARKERS}
want_ens |= {SYM_TO_ENS.get(s.title()) for s in MARKERS}
want_ens = {str(x) for x in want_ens if x}

panel_rows = []
for (name, ct, tp), df in effects.items():
    sub = df[
        df["ensembl_id"].astype(str).isin(want_ens)
        | df["gene_symbol"].astype(str).str.upper().isin(MARKERS)
    ].copy()
    for _, r in sub.iterrows():
        panel_rows.append({
            "experiment": exp_lab(name),
            "name": name,
            "cell_type": ct,
            "time": tp.replace("_LPS", ""),
            "gene": str(r["gene_symbol"]).upper(),
            "delta": r["delta"],
            "log2fc": r["log2fc"],
            "fdr": r["fdr"],
            "mean_pert": r["mean_pert"],
            "mean_ctrl": r["mean_ctrl"],
        })
panel = pd.DataFrame(panel_rows)
display(panel.sort_values(["cell_type", "time", "gene", "experiment"]).round(4))
panel.to_csv(OUT_DIR / "cytokine_marker_deltas_by_celltype.csv", index=False)

# 6h pivot gene x experiment for each CT
for ct in FOCUS_CTS:
    sub = panel[(panel["cell_type"] == ct) & (panel["time"] == "6h")]
    if sub.empty:
        print("no marker rows for", ct)
        continue
    piv = sub.pivot_table(index="gene", columns="experiment", values="delta")
    print(f"\n=== {ct} @6h marker Δ ===")
    display(piv.round(4))


## 6. JEPA vs MaskGIT agreement (numbers)

Per cell type @ each time: Spearman of per-gene Δ, sign agreement, frac(Δ>0), mean|Δ|.
Default compare: each JEPA setting vs `reference` (MaskGIT).


In [ ]:
def join_pair(name_a: str, name_b: str, ct: str, tp: str) -> pd.DataFrame | None:
    key_a, key_b = (name_a, ct, tp), (name_b, ct, tp)
    if key_a not in effects or key_b not in effects:
        return None
    a = effects[key_a][["ensembl_id", "gene_symbol", "delta"]].rename(
        columns={"delta": f"delta_{name_a}"}
    )
    b = effects[key_b][["ensembl_id", "delta"]].rename(columns={"delta": f"delta_{name_b}"})
    return a.merge(b, on="ensembl_id", how="inner")


sum_rows = []
others = [n for n in EXPS if n != REF_NAME]
for ct in FOCUS_CTS:
    for tp in TIMES:
        for name in others:
            m = join_pair(REF_NAME, name, ct, tp)
            if m is None or len(m) < 50:
                continue
            xa, xb = f"delta_{REF_NAME}", f"delta_{name}"
            both = (m[xa].abs() > 1e-6) & (m[xb].abs() > 1e-6)
            agree = float((np.sign(m.loc[both, xa]) == np.sign(m.loc[both, xb])).mean()) if both.any() else np.nan
            sum_rows.append({
                "cell_type": ct,
                "time": tp.replace("_LPS", ""),
                "a": exp_lab(REF_NAME),
                "b": exp_lab(name),
                "n_shared": len(m),
                "spearman": stats.spearmanr(m[xa], m[xb]).correlation,
                "pearson": stats.pearsonr(m[xa], m[xb])[0],
                "sign_agree": agree,
                "frac_up_b": float((m[xb] > 0).mean()),
                "frac_up_a": float((m[xa] > 0).mean()),
                "mean_abs_delta_b": float(m[xb].abs().mean()),
                "mean_abs_delta_a": float(m[xa].abs().mean()),
            })
sum_df = pd.DataFrame(sum_rows)
display(sum_df.round(4))
sum_df.to_csv(OUT_DIR / "delta_agreement_by_celltype.csv", index=False)


## 7. Top 10 up / down (JEPA src mask vs MaskGIT)

One table per focus cell type @ 6h.


In [ ]:
for ct in FOCUS_CTS:
    for name in ("maskgit", "jepa_srconly_mask"):
        key = (name, ct, "6h_LPS")
        if key not in effects:
            print("missing", key)
            continue
        df = effects[key]
        print(f"\n===== {exp_lab(name)} | {ct} @6h =====")
        print("DOWN:")
        display(df.nsmallest(10, "delta")[["gene_symbol", "delta", "mean_ctrl", "mean_pert", "fdr"]].round(4))
        print("UP:")
        display(df.nlargest(10, "delta")[["gene_symbol", "delta", "mean_ctrl", "mean_pert", "fdr"]].round(4))


## 8. Optional pad vs mask (same count-input)

Numerical agreement only (no scatter grid).


In [ ]:
pair_rows = []
for ct in FOCUS_CTS:
    for tp in TIMES:
        for a, b in PAIRS:
            m = join_pair(a, b, ct, tp)
            if m is None:
                continue
            xa, xb = f"delta_{a}", f"delta_{b}"
            pair_rows.append({
                "cell_type": ct,
                "time": tp.replace("_LPS", ""),
                "a": exp_lab(a),
                "b": exp_lab(b),
                "spearman": stats.spearmanr(m[xa], m[xb]).correlation,
                "mean_abs_a": float(m[xa].abs().mean()),
                "mean_abs_b": float(m[xb].abs().mean()),
            })
pair_df = pd.DataFrame(pair_rows)
display(pair_df.round(4))
pair_df.to_csv(OUT_DIR / "pad_vs_mask_agreement_by_celltype.csv", index=False)


## 9. How to read

- NKT has very few unique cells (~16) — noisy.
- gamma-delta is not in these dumps; CD8+ is the substitute third type.
- Prefer ranks / signs / Spearman over raw Δ magnitude across model families.


## 10. Re-run inference (optional)

Pinned qresync dumps in YAMLs. `RERUN_INFERENCE = False`.


In [ ]:
def cli_cmd(cfg: dict) -> list[str]:
    c = cfg["cli"]
    return ["python", c["script"], *list(map(str, c["args"]))]

for name, cfg in EXPS.items():
    print(f"\n# {name}  {cfg['experiment']['label']}")
    print(" ".join(cli_cmd(cfg)))
print("\nRERUN_INFERENCE=False — not launching." if not RERUN_INFERENCE else "would launch")


## 11. Done

Outputs under `OUT_DIR`.


In [ ]:
print("outputs:", OUT_DIR)
print(sorted(p.name for p in OUT_DIR.glob("*")))
